# Generate Simulated Dynamic Homogeneous Link Data

This notebook creates snapshot data for `DyMLH-Link`. The intended setting is:

```text
2015, 2016, 2017, 2018, 2019 snapshots -> predict 2020 links
```

Each yearly graph is a homogeneous DGL graph. Historical snapshots contain node features and `global_id`. The target graph, for example `graph_2020.bin`, additionally stores edge-level `train_mask`, `valid_mask`, and `test_mask`.

In [ ]:
import os
from pathlib import Path

import numpy as np
import torch
import dgl

SEED = 42
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

OUT_DIR = Path('data/simulated')
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEARS = list(range(2015, 2021))
HISTORY_YEARS = YEARS[:-1]
TARGET_YEAR = YEARS[-1]

NUM_GLOBAL_NODES = 1000
FEAT_DIM = 32
LATENT_DIM = 16

# Nodes may appear/disappear across years. Increase this if every year should contain almost all nodes.
BASE_ACTIVE_PROB = 0.78
NEW_NODE_DRIFT = 0.025

# Controls graph density. Larger values create more edges.
EDGE_SCALE = 0.12
MAX_CANDIDATE_PAIRS = 180_000

TRAIN_RATIO = 0.7
VALID_RATIO = 0.1
TEST_RATIO = 0.2

print('Output directory:', OUT_DIR.resolve())

## Helper Functions

The simulation uses evolving latent node vectors. Edges are sampled from pairwise latent similarity, so links are temporally correlated but not identical across years.

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def active_nodes_for_year(year_idx):
    active_prob = min(0.98, BASE_ACTIVE_PROB + NEW_NODE_DRIFT * year_idx)
    active = rng.random(NUM_GLOBAL_NODES) < active_prob
    # Keep a stable core so temporal models always have overlap.
    active[: int(NUM_GLOBAL_NODES * 0.35)] = True
    return np.flatnonzero(active).astype(np.int64)


def sample_edges(global_ids, latent):
    n = len(global_ids)
    if n < 2:
        return np.array([], dtype=np.int64), np.array([], dtype=np.int64)

    # Candidate pairs are sampled first to avoid building an n x n dense matrix.
    num_candidates = min(MAX_CANDIDATE_PAIRS, n * (n - 1))
    src = rng.integers(0, n, size=num_candidates, dtype=np.int64)
    dst = rng.integers(0, n, size=num_candidates, dtype=np.int64)
    keep = src != dst
    src, dst = src[keep], dst[keep]

    score = np.sum(latent[src] * latent[dst], axis=1) / np.sqrt(latent.shape[1])
    prob = EDGE_SCALE * sigmoid(score)
    chosen = rng.random(len(prob)) < prob
    src, dst = src[chosen], dst[chosen]

    if len(src) == 0:
        return src, dst

    # Remove duplicate directed edges.
    pairs = np.unique(np.stack([src, dst], axis=1), axis=0)
    return pairs[:, 0].astype(np.int64), pairs[:, 1].astype(np.int64)


def make_masks(num_edges):
    order = rng.permutation(num_edges)
    n_train = int(num_edges * TRAIN_RATIO)
    n_valid = int(num_edges * VALID_RATIO)
    train_idx = order[:n_train]
    valid_idx = order[n_train:n_train + n_valid]
    test_idx = order[n_train + n_valid:]

    train_mask = torch.zeros(num_edges, dtype=torch.bool)
    valid_mask = torch.zeros(num_edges, dtype=torch.bool)
    test_mask = torch.zeros(num_edges, dtype=torch.bool)
    train_mask[torch.as_tensor(train_idx, dtype=torch.long)] = True
    valid_mask[torch.as_tensor(valid_idx, dtype=torch.long)] = True
    test_mask[torch.as_tensor(test_idx, dtype=torch.long)] = True
    return train_mask, valid_mask, test_mask

## Generate and Save Snapshot Graphs

The target year graph stores the split masks on its positive edges. Historical years do not need edge masks.

In [ ]:
base_latent = rng.normal(0, 1, size=(NUM_GLOBAL_NODES, LATENT_DIM)).astype(np.float32)
feature_projection = rng.normal(0, 0.6, size=(LATENT_DIM, FEAT_DIM)).astype(np.float32)

saved_paths = []
for year_idx, year in enumerate(YEARS):
    global_ids = active_nodes_for_year(year_idx)

    temporal_trend = 0.08 * year_idx
    latent_noise = rng.normal(0, 0.12, size=(len(global_ids), LATENT_DIM)).astype(np.float32)
    latent = base_latent[global_ids] + temporal_trend + latent_noise

    feat_noise = rng.normal(0, 0.05, size=(len(global_ids), FEAT_DIM)).astype(np.float32)
    features = latent @ feature_projection + feat_noise

    src, dst = sample_edges(global_ids, latent)
    graph = dgl.graph((torch.from_numpy(src), torch.from_numpy(dst)), num_nodes=len(global_ids))
    graph.ndata['feat'] = torch.tensor(features, dtype=torch.float32)
    graph.ndata['global_id'] = torch.tensor(global_ids, dtype=torch.long)

    if year == TARGET_YEAR:
        train_mask, valid_mask, test_mask = make_masks(graph.num_edges())
        graph.edata['train_mask'] = train_mask
        graph.edata['valid_mask'] = valid_mask
        graph.edata['test_mask'] = test_mask

    path = OUT_DIR / f'graph_{year}.bin'
    dgl.save_graphs(str(path), [graph], {'year': torch.tensor([year])})
    saved_paths.append(path)
    print(year, graph, 'saved to', path)

## Inspect the Target Graph

In [ ]:
target_graph, metadata = dgl.load_graphs(str(OUT_DIR / f'graph_{TARGET_YEAR}.bin'))
g = target_graph[0]
print(g)
print('metadata:', metadata)
print('node data:', list(g.ndata.keys()))
print('edge data:', list(g.edata.keys()))
print('train/valid/test edges:', int(g.edata['train_mask'].sum()), int(g.edata['valid_mask'].sum()), int(g.edata['test_mask'].sum()))

## Training Command

After running the cells above, use this command from the repository root.

In [ ]:
snapshot_bins = ','.join(str(OUT_DIR / f'graph_{year}.bin') for year in HISTORY_YEARS)
target_bin = str(OUT_DIR / f'graph_{TARGET_YEAR}.bin')

cmd = f'''python -u Link_Prediction.py \
  --snapshot-bins {snapshot_bins} \
  --target-bin {target_bin} \
  --feat-key feat \
  --global-id-key global_id \
  --hidden-dim 128 \
  --gnn-layers 2 \
  --sage-aggregator-type mean \
  --temporal-model gru \
  --temporal-layers 1 \
  --predictor dot \
  --negative-ratio 1.0 \
  --eval-negative-ratio 1.0 \
  --epochs 200 \
  --patience 30 \
  --early-stop-metric auc \
  --log-every 10 \
  --output-dir outputs'''
print(cmd)